# Simple recursive network for sequence learning

In [1]:
import sys
from pathlib import Path
import numpy as np
import torch
from typing import Callable, Sequence

# import local packages
sys.path.append(str(Path().resolve().parents[0]))

from src.utils.logger import setup_logger, add_log_level
from src.utils.args import get_args
from src.data.formater import PandasLoader


In [2]:
args = get_args()
logger_level = "TRACE"


In [3]:
add_log_level("TRACE", 5)
logger = setup_logger(name="logger", level=logger_level)
logger.info("logger set to info level")
logger.trace("TRACE logger activated")



INFO - logger set to info level
TRACE - TRACE logger activated


In [13]:
loader = PandasLoader(args["input"], args["format"])
responses = loader.get()
logger.info(f"subjects response per trial: \n {responses}")
# converting to array of shape (subjects, responses)
responses = responses.to_numpy().T
logger.info(f"responses after transposition= {responses}")
# responses = np.expand_dims(responses, axis=-1)
# logger.trace(f"response after expansions on last dim={responses}")
# get set of responses values and their order of appearance(giving them a numerical id)
unique_vals, encoded = np.unique(responses.reshape(-1), return_inverse=True)
logger.info(f"unique values (set)={unique_vals}")
logger.info(f"encoded values={encoded}")
encoded = encoded.reshape(responses.shape)
logger.debug(f"final encoded shape (subjests, responses)={encoded.shape}")



INFO - subjects response per trial: 
 Subject    1  2  3  4  5  6  7  8  9  10  ... 51 52 53 54 55 56 57 58 59 60
TrialIndex                                ...                              
0           a  a  a  b  c  a  b  c  a  a  ...  a  a  a  b  c  a  b  c  b  b
1           J  Q  P  M  T  K  I  S  J  Q  ...  J  Q  P  M  T  K  I  S  L  H
2           d  d  d  e  f  d  e  f  d  d  ...  d  d  d  e  f  d  e  f  e  e
3           a  b  b  c  c  b  c  b  a  b  ...  a  b  b  c  c  b  c  b  b  c
4           S  I  O  Y  I  £  Q  H  S  I  ...  S  I  O  Y  I  £  Q  H  K  K
...        .. .. .. .. .. .. .. .. .. ..  ... .. .. .. .. .. .. .. .. .. ..
643         P  N  H  Y  £  G  H  M  P  N  ...  P  N  H  Y  £  G  H  M  H  W
644         d  e  d  e  d  d  f  f  d  e  ...  d  e  d  e  d  d  f  f  f  f
645         c  c  c  c  a  a  c  b  c  c  ...  c  c  c  c  a  a  c  b  b  c
646         O  I  K  ù  I  ù  &  Z  O  I  ...  O  I  K  ù  I  ù  &  Z  N  H
647         f  f  f  f  d  d  f  e  f  f  ...  f  

In [ ]:
def make_uniform_tensor(
    extremum: tuple[float, float], shape: Sequence[int], grad: bool
):
    return extremum[0] + (extremum[1] - extremum[0]) * torch.rand(
        size=shape, requires_grad=grad
    )



![test](https://web.stanford.edu/group/pdplab/pdphandbook/srn_net.png)

A SRN is a simplified RNN. The output of the hidden layer is fed back as input to the hidden layer at the
next time step. The output of the hidden layer is also used to compute the output of the network.

![SRN basic architecture](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.researchgate.net%2Fpublication%2F361380872%2Ffigure%2Ffig1%2FAS%3A1169080920875058%401655742020827%2FSchematic-diagram-of-Elman-network-structure-in-simple-recurrent-neural-network.jpg&f=1&nofb=1&ipt=1d5a36f3ef48ba69e88b9f96a9176f2e1ed8232b2019b9d3f7f7335e1ee85f1d)
```mermaid
graph TB;
hidden --> context
input --> hidden
context --> hidden
hidden --> output
```

```mermaid
graph TB;
x1 & x2 --> h1 & h2 --> y1 & y2
c1 & c2 --> h1 & h2
```

In [ ]:


class SRN(AbstractNNModel):
    def __init__(self, params):
        self.allowed_parameters = {
            "vocab_size": int,
            "hidden_size": int,
            "lr": float,
            "mu": float,
            "clearval": float,
            "epochs": int,
        }
        self.init_model(params)

    def validate_parameters(self, params):
        for parameter, value in params.items():
            if parameter not in self.allowed_parameters.keys():
                raise ParameterNotAllowedException(parameter)
            expected_type = self.allowed_parameters[parameter]
            if not isinstance(value, expected_type):
                raise WrongParameterTypeException(
                    parameter, value.type(), expected_type
                )
        for parameter in self.allowed_parameters:
            if parameter not in params.key():
                raise MissingParameterException(parameter)
        return True

    def init_model(self, params):

        self.validate_parameters(params)
        self.input_size = params["vocab_size"]
        self.hidden_size = params["hidden_size"]
        self.output_size = params["vocab_size"]

        self.lr = params["lr"]
        self.mu = params["mu"]
        self.clearval = params["clearval"]

        self.Wxh = np.random.uniform(-0.1, 0.1, (self.hidden_size, self.output_size))
        self.Whh = np.random.uniform(-0.1, 0.1, (self.hidden_size, self.hidden_size))
        self.Why = np.random.uniform(-0.1, 0.1, (self.output_size, self.hidden_size))

        self.bh = np.zeros(self.hidden_size)
        self.by = np.zeros(self.output_size)

        self.reset_context()

    def reset_context(self):
        self.context = np.ones(self.hidden_size) * self.clearval

    def softmax(self, x):
        e = np.exp(x - np.max(x))
        return e / np.sum(e)

    def forward(self, x):

        h = np.tanh(self.Wxh @ x + self.Whh @ self.context + self.bh)

        y = self.softmax(self.Why @ h + self.by)

        return h, y

    def train_step(self, x, target):

        h, y = self.forward(x)

        dy = y - target

        dWhy = np.outer(dy, h)
        dby = dy

        dh = self.Why.T @ dy
        dh_raw = (1 - h**2) * dh

        dWxh = np.outer(dh_raw, x)
        dWhh = np.outer(dh_raw, self.context)
        dbh = dh_raw

        self.Why -= self.lr * dWhy
        self.by -= self.lr * dby

        self.Wxh -= self.lr * dWxh
        self.Whh -= self.lr * dWhh
        self.bh -= self.lr * dbh

        self.context = h + self.mu * self.context

        loss = -np.sum(target * np.log(y + 1e-12))
        return loss